# Clinical Decision Support: Dosage to Mastery Forecasting (Optimized)
### Strategic Objective
This model predicts the **total number of practice trials (Dosage)** a patient needs to reach a stable state of word recovery. By moving beyond simple binary success/failure, we provide clinicians with a 'Time-to-Mastery' estimate for better session planning.

## 1. Data Preprocessing
In this stage, we merge naming probes, treatment trials (primed and unprimed), and outcome metadata into a unified patient timeline.
**Key Steps:**
1. **Cleaning**: Standardizing player IDs and handling inconsistent date formats.
2. **Target Definition**: Mastery is strictly defined as the first occurrence of **3 consecutive successes** for a specific word.
3. **Aggregation**: We pivot from trial-level (one row per attempt) to item-level (one row per word-patient pair) to calculate the total dosage.

In [ ]:
# Load evolved libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, r2_score
from scipy.stats import linregress

sns.set_theme(style='whitegrid', context='talk')
plt.rcParams['figure.figsize'] = (14, 7)

# LOAD RAW DATA
probes = pd.read_csv('1. 2019-11-12_naming_probes_tidy.csv')
tx_nop = pd.read_csv('2. 2019-10-25_treatment_retrieval_noprime.csv')
tx_p = pd.read_csv('4. 2019-10-25_treatment_retrieval_prime.csv')
outcomes = pd.read_csv('6. 2019-10-3_Outcome_data_tidy.csv')

for d in [probes, tx_nop, tx_p, outcomes]:
    d['player'] = d['player'].str.lower()

p_clean = probes[['player', 'date', 'target', 'trial_resp.corr.hand', 'vocal.rt', 'complexity_score', 'phase']].rename(columns={'target':'word', 'trial_resp.corr.hand':'success'})
nop_clean = tx_nop[['player', 'date', 'stim_text', 'naming1_resp.corr', 'naming1_vocal.rt']].rename(columns={'stim_text':'word', 'naming1_resp.corr':'success', 'naming1_vocal.rt':'vocal.rt'})
p_clean_tx = tx_p[['player', 'date', 'stim_text', 'naming2_resp.corr', 'naming2_vocal.rt']].rename(columns={'stim_text':'word', 'naming2_resp.corr':'success', 'naming2_vocal.rt':'vocal.rt'})

df = pd.concat([p_clean, nop_clean, p_clean_tx])
df['date'] = pd.to_datetime(df['date'].str.replace('_', '-'), errors='coerce', format='mixed')
df = df.sort_values(['player', 'word', 'date']).dropna(subset=['success'])
df['trial_seq'] = df.groupby(['player', 'word']).cumcount() + 1

# Target Calculation: Identify when the 3-success streak happened
def get_trials_to_mastery(group):
    streak = 0
    for i, row in enumerate(group.itertuples()):
        if row.success == 1: streak += 1
        else: streak = 0
        if streak == 3: return row.trial_seq
    return np.nan

dosage_df = df.groupby(['player', 'word']).apply(get_trials_to_mastery).reset_index()
dosage_df.columns = ['player', 'word', 'trials_to_mastery']
dosage_df = dosage_df.dropna()
print(f'Preprocessing complete. Analyzable words: {len(dosage_df)}')

## 2. Feature Engineering: Behavioral Dynamics
Instead of just using averages, we extract the **dynamics** of early learning within the observation window.

**Features Engineered:**
*   **Mean Accuracy/Speed**: The baseline performance levels.
*   **Acquisition Slope**: The linear trend of performance (is it getting better or worse during the first few trials?).
*   **RT Jitter**: Retrieval Stability—measured as the Coefficient of Variation (StdDev/Mean). High jitter indicates fragile recovery.

In [ ]:
def calculate_slope(y):
    if len(y) < 2: return 0
    x = np.arange(len(y))
    slope, _, _, _, _ = linregress(x, y)
    return slope if not np.isnan(slope) else 0

# Aggregate clinical demos (Age/Time Since Stroke)
demos = outcomes.groupby('player').agg({'demo.mpo': 'first', 'demo.age': 'first'}).reset_index()

## 3. Training Set Preview: Inputs & Outputs
Here we show the first 5 rows of the data that will be fed into the AI model. 

### Model Variables:
*   **Inputs (Features)**: `acc_mean`, `acc_slope` (Momentum), `rt_mean`, `rt_jitter` (Stability), `complexity`, `demo.mpo`, `demo.age`.
*   **Output (Target)**: `trials_to_mastery` (Log-transformed during training for higher precision).

### First 5 Rows (Observation window N=12):

In [ ]:
# Extract features for the 12-trial window for preview
N = 12
early_subset = df[df['trial_seq'] <= N]
early_features = early_subset.groupby(['player', 'word']).agg({
    'success': ['mean', calculate_slope], 
    'vocal.rt': ['mean', 'std', calculate_slope],
    'complexity_score': 'first'
}).reset_index()
early_features.columns = ['player', 'word', 'acc_mean', 'acc_slope', 'rt_mean', 'rt_std', 'rt_slope', 'complexity']
early_features['rt_jitter'] = early_features['rt_std'] / (early_features['rt_mean'] + 1e-9)

preview_data = dosage_df.merge(early_features, on=['player', 'word']).merge(demos, on='player', how='left').fillna(0)
preview_data.head(5)

## 4. Modeling Strategy
### Algorithm: Gradient Boosting Regressor
We use a **Gradient Boosting Regressor**, which builds an additive model in a forward stage-wise fashion. This allows the model to capture non-linear relationships, such as how 'Age' might interact specifically with 'RT Jitter' to increase dosage.

### Mathematical Refinement: Log-Scaling
Patients needing long-term practice (outliers) can skew the results. We train on the **Logarithm** of the dosage, which prioritizes accuracy for the majority of patients while remaining robust to outliers.

In [ ]:
# Optimized Training Loop (Sensitivity Analysis)
results = []
windows = [1, 3, 5, 8, 12]

for N in windows:
    early_subset = df[df['trial_seq'] <= N]
    early_features = early_subset.groupby(['player', 'word']).agg({
        'success': ['mean', calculate_slope], 
        'vocal.rt': ['mean', 'std', calculate_slope],
        'complexity_score': 'first'
    }).reset_index()
    early_features.columns = ['player', 'word', 'acc_mean', 'acc_slope', 'rt_mean', 'rt_std', 'rt_slope', 'complexity']
    early_features['rt_jitter'] = early_features['rt_std'] / (early_features['rt_mean'] + 1e-9)
    
    data = dosage_df.merge(early_features, on=['player', 'word']).merge(demos, on='player', how='left').fillna(0)
    
    X = data[['acc_mean', 'acc_slope', 'rt_mean', 'rt_jitter', 'rt_slope', 'complexity', 'demo.mpo', 'demo.age']]
    y = np.log1p(data['trials_to_mastery'])
    
    players = data['player'].unique()
    train_p = players[:int(len(players)*0.8)]
    X_train, y_train = X[data['player'].isin(train_p)], y[data['player'].isin(train_p)]
    X_test, y_test = X[~data['player'].isin(train_p)], y[~data['player'].isin(train_p)]
    
    model = GradientBoostingRegressor(n_estimators=100, random_state=42)
    model.fit(X_train, y_train)
    preds = np.expm1(model.predict(X_test))
    actuals = np.expm1(y_test)
    
    mae = mean_absolute_error(actuals, preds)
    results.append({'Window': f'{N} Tr.', 'MAE': mae, 'y_test': actuals, 'y_pred': preds, 'model': model, 'features': X.columns})
    print(f'Window: {N:2d} - Optimized MAE: {mae:.2f} trials')

## 5. Optimized Performance Overview
Visualizing the Accuracy Gain and the Clinical Confidence Center.

In [ ]:
# Results Visualization
plt.figure(figsize=(12, 6))
sns.barplot(x=[r['Window'] for r in results], y=[r['MAE'] for r in results], palette='magma')
plt.title('Model Accuracy vs. Observation Period (MAE)')
plt.ylabel('Error (Avg. Trials)')
plt.show()

# Clinical Safety Zone (N=12)
res12 = results[-1]
yt, yp = res12['y_test'], res12['y_pred']

# Calculate bounds
within_5 = np.abs(yt - yp) <= 5
accuracy_pct = np.mean(within_5) * 100

plt.figure(figsize=(12, 9))
sns.scatterplot(x=yt, y=yp, hue=within_5, palette={True: '#2ecc71', False: '#e74c3c'}, 
                s=100, alpha=0.7, edgecolor='w')

# Identity Line & Safety Zone
plt.plot([0, 60], [0, 60], '--', color='#34495e', alpha=0.8, label='Perfect Prediction')
plt.fill_between([0, 60], [0-5, 60-5], [0+5, 60+5], color='#2ecc71', alpha=0.1, 
                 label='Clinical Safety Zone (±5 Trials)')

# Annotations
plt.text(5, 50, f'Clinical Success: {accuracy_pct:.1f}% within bound', 
         fontsize=14, fontweight='bold', bbox=dict(facecolor='white', alpha=0.8))
plt.text(45, 5, 'Under-predicting\n(Risk of premature discharge)', color='#c0392b', fontsize=12, alpha=0.7)
plt.text(5, 55, 'Over-predicting\n(Risk of over-training)', color='#2980b9', fontsize=12, alpha=0.7)

plt.title('Final Clinical Performance: Reality vs. AI Prediction (12 Trials observed)')
plt.xlabel('Actual Number of Trials needed for Mastery')
plt.ylabel('AI Predicted Number of Trials')
plt.legend(loc='lower right')
plt.grid(True, linestyle=':', alpha=0.6)
plt.xlim(0, 60)
plt.ylim(0, 60)
plt.show()